# 08. 저장된 예측으로 모델 가치·의사결정 분석

05·07의 **기존 Valid/Test 예측 파일만** 읽습니다. 학습, Feature 재생성, Git 갱신은 하지 않습니다. 기준일 `t`에서 `t+1`~`t+7`의 동일 매장·상품군·목표일을 비교하고, 전주 동일 요일·최근 4주 동일 요일 평균을 기준선으로 둡니다.

Test는 과거 실험에서 이미 확인한 회고적 구간입니다. 여기서 모델 설정을 다시 선택하지 않습니다. 실제 물량 배분·발주 효과는 재고와 공급 자료가 없으므로 측정하지 않습니다.


## Colab 준비

별도의 00 노트북으로 저장소가 My Drive에 준비되어 있어야 합니다. 원본 CSV나 GPU 런타임은 이 분석에 필요하지 않습니다.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/retail-demand-forecasting


### 시각화 패키지

모델을 설치하거나 학습하지 않습니다. 결과 Parquet과 한글 그래프를 읽고 그리는 패키지만 준비합니다.


In [ ]:
%pip install -q seaborn koreanize-matplotlib pyarrow


## Library


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import koreanize_matplotlib

REPO = Path("/content/drive/MyDrive/retail-demand-forecasting")
assert (REPO / "src/retail_forecast/__init__.py").is_file(), REPO
sys.path.insert(0, str(REPO / "src"))

from retail_forecast.comparison import compare_on_common_rows, load_common_predictions
from retail_forecast.metrics import regression_metrics
from retail_forecast.neural_data import KEYS, summarize_neural_predictions

sns.set_theme(style="whitegrid", font="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False


## Utils

공통 행 비교, 기준선 추가, 그룹별 오류 계산에 필요한 짧은 함수만 둡니다.


In [ ]:
def prediction_paths(ml_dir, dl_dir, phase):
    return {
        "LightGBM": ml_dir / f"{phase}_predictions_lightgbm.csv",
        "CatBoost": ml_dir / f"{phase}_predictions_catboost.csv",
        "NHITS": dl_dir / "nhits" / f"{phase}_predictions_nhits.parquet",
        "TCN": dl_dir / "tcn" / f"{phase}_predictions_tcn.parquet",
        "TFT": dl_dir / "tft" / f"{phase}_predictions_tft.parquet",
    }


def compare_with_baselines(paths):
    model_metrics = compare_on_common_rows(paths)
    reference = pd.read_csv(paths["LightGBM"], parse_dates=["date", "target_date"])
    baselines = {
        "전주 동일 요일": "sales_same_weekday_last_week",
        "최근 4주 동일 요일 평균": "sales_same_weekday_4w_mean",
    }
    baseline_metrics = []
    for name, column in baselines.items():
        frame = reference[KEYS + ["target_sales", column]].rename(columns={column: "prediction"})
        metrics, _ = summarize_neural_predictions(frame, name)
        baseline_metrics.append(metrics)
    return pd.concat([model_metrics, *baseline_metrics], ignore_index=True)


def group_scores(frame, groups):
    errors = frame.assign(
        model_absolute=(frame["prediction"] - frame["target_sales"]).abs(),
        baseline_absolute=(frame["sales_same_weekday_4w_mean"] - frame["target_sales"]).abs(),
        model_signed=frame["prediction"] - frame["target_sales"],
    )
    result = errors.groupby(groups, observed=True).agg(
        rows=("target_sales", "size"),
        actual_sales=("target_sales", "sum"),
        model_absolute=("model_absolute", "sum"),
        baseline_absolute=("baseline_absolute", "sum"),
        model_signed=("model_signed", "sum"),
    ).reset_index()
    denominator = result["actual_sales"].replace(0, np.nan)
    result["lightgbm_wape"] = 100 * result["model_absolute"] / denominator
    result["baseline_wape"] = 100 * result["baseline_absolute"] / denominator
    result["gain_pp"] = result["baseline_wape"] - result["lightgbm_wape"]
    result["excess_absolute_error"] = result["model_absolute"] - result["baseline_absolute"]
    result["actual_share_pct"] = 100 * result["actual_sales"] / result["actual_sales"].sum()
    result["lightgbm_bias"] = 100 * result["model_signed"] / denominator
    return result.drop(columns=["model_absolute", "baseline_absolute", "model_signed"])


def weekly_predictions(frame):
    keys = ["date", "store_nbr", "family"]
    assert frame.groupby(keys, observed=True)["forecast_horizon"].nunique().eq(7).all()
    return frame.groupby(keys, observed=True)[
        ["target_sales", "prediction", "sales_same_weekday_4w_mean"]
    ].sum(min_count=7).reset_index()


def assign_demand_tier(frame, cutoffs):
    result = frame.copy()
    demand = result["sales_same_weekday_4w_mean"]
    result["demand_tier"] = pd.Categorical(
        np.select([demand.le(cutoffs[0]), demand.le(cutoffs[1])],
                  ["낮음", "중간"], default="높음"),
        categories=["낮음", "중간", "높음"], ordered=True,
    )
    return result


def model_weekly_predictions(paths, reference, cutoffs):
    baseline = reference[KEYS + ["sales_same_weekday_4w_mean"]]
    results = {}
    for name, path in paths.items():
        predictions = load_common_predictions(path).merge(
            baseline, on=KEYS, validate="one_to_one",
        )
        assert len(predictions) == len(reference), f"{name}: 평가 행이 다릅니다."
        results[name] = assign_demand_tier(weekly_predictions(predictions), cutoffs)
    return results


def model_group_scores(weekly_by_model, groups):
    tables = []
    for name, weekly in weekly_by_model.items():
        table = group_scores(weekly, groups).rename(columns={
            "lightgbm_wape": "model_wape", "lightgbm_bias": "model_bias",
        })
        table.insert(0, "model", name)
        tables.append(table)
    return pd.concat(tables, ignore_index=True)


def model_fallback_scores(valid_by_model, test_by_model):
    rows = []
    for name, valid in valid_by_model.items():
        valid_low = valid.loc[valid["demand_tier"].eq("낮음")]
        model_error = (valid_low["prediction"] - valid_low["target_sales"]).abs().sum()
        baseline_error = (
            valid_low["sales_same_weekday_4w_mean"] - valid_low["target_sales"]
        ).abs().sum()
        use_fallback = bool(model_error > baseline_error)
        for phase, frame in [("Valid", valid), ("Test", test_by_model[name])]:
            fallback = frame["demand_tier"].eq("낮음") & use_fallback
            forecast = frame["prediction"].where(
                ~fallback, frame["sales_same_weekday_4w_mean"],
            )
            for method, prediction in [("모델", frame["prediction"]),
                                       ("4주 기준선", frame["sales_same_weekday_4w_mean"]),
                                       ("낮은 수요 대체", forecast)]:
                rows.append({
                    "phase": phase, "model": name, "method": method,
                    "rows": len(frame), "low_demand_fallback": use_fallback,
                    **regression_metrics(frame["target_sales"], prediction),
                })
    return pd.DataFrame(rows)


## 결과 경로

05와 07에서 사용한 실행 이름을 그대로 적습니다. 필요한 예측 파일이 없으면 해당 실행의 **결과 파일**을 먼저 준비하세요. 이 노트북은 학습을 대신 실행하지 않습니다.


In [ ]:
ML_RUN_NAME = "full_history_time_split_fixed_i600"
DL_RUN_NAME = "neural_time_split_val_loss"

ml_dir = REPO / "outputs" / ML_RUN_NAME
dl_dir = REPO / "outputs" / DL_RUN_NAME
analysis_dir = REPO / "outputs" / "decision_analysis" / f"{ML_RUN_NAME}__{DL_RUN_NAME}"
analysis_dir.mkdir(parents=True, exist_ok=True)

valid_paths = prediction_paths(ml_dir, dl_dir, "validation")
test_paths = prediction_paths(ml_dir, dl_dir, "test")
missing = [path for path in [*valid_paths.values(), *test_paths.values()] if not path.is_file()]
assert not missing, f"필요한 결과 파일이 없습니다: {missing}"


## 1. 동일한 행에서 모델·기준선 비교

Valid와 Test를 별도로 계산합니다. `compare_on_common_rows`가 다섯 모델의 기준일·목표일·매장·상품군·horizon과 정답이 일치하는지 검사합니다. 양쪽 기준선은 LightGBM 결과 파일에 저장된 같은 행의 과거 판매값을 사용합니다.


In [ ]:
valid_table = compare_with_baselines(valid_paths)
test_table = compare_with_baselines(test_paths)

valid_table.to_csv(analysis_dir / "comparison_with_baselines_valid.csv", index=False)
test_table.to_csv(analysis_dir / "comparison_with_baselines_test.csv", index=False)

overall = pd.concat([
    valid_table.assign(phase="Valid"),
    test_table.assign(phase="Test"),
], ignore_index=True)
overall = overall.loc[
    overall["forecast_horizon"].eq(0),
    ["phase", "model", "scope", "rows", "mae", "rmse", "wape", "bias"],
].sort_values(["phase", "scope", "wape"])
display(overall)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(
    data=overall.loc[overall["phase"].eq("Test")],
    x="wape", y="model", hue="scope", ax=ax,
)
ax.set(xlabel="WAPE (%)", ylabel="", title="동일 Test 행: 모델과 주간 기준선")
fig.tight_layout()
fig.savefig(analysis_dir / "test_wape_comparison.png", dpi=160)
plt.show()


## 2. LightGBM이 기준선보다 나쁜 구간 찾기

`gain_pp = 기준선 WAPE − LightGBM WAPE`입니다. **음수**이면 LightGBM이 최근 4주 평균보다 나쁩니다. `actual_sales`가 작은 집단에서는 WAPE가 불안정할 수 있으므로 행 수와 판매량을 함께 봅니다. 아래 표는 Test를 해석하기 위한 것이며 Test 결과로 파라미터를 조정하지 않습니다.


In [ ]:
test_predictions = pd.read_csv(test_paths["LightGBM"], parse_dates=["date", "target_date"])

horizon_scores = group_scores(test_predictions, ["forecast_horizon"])
family_scores = group_scores(test_predictions, ["family"])
store_scores = group_scores(test_predictions, ["store_nbr"])

for name, table in {
    "horizon": horizon_scores,
    "family": family_scores,
    "store": store_scores,
}.items():
    table.to_csv(analysis_dir / f"test_error_by_{name}.csv", index=False)

display(horizon_scores)
display(family_scores.sort_values("gain_pp").head(12))
display(store_scores.sort_values("gain_pp").head(12))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="gain_pp", marker="o", ax=axes[0])
sns.lineplot(data=horizon_scores, x="forecast_horizon", y="lightgbm_bias", marker="o", ax=axes[1])
axes[0].axhline(0, color="gray", linewidth=1)
axes[0].set(xlabel="예측 거리 (일)", ylabel="WAPE 개선 (pp)", title="4주 기준선 대비 개선")
axes[1].set(xlabel="예측 거리 (일)", ylabel="Bias (%)", title="LightGBM 과대·과소예측")
fig.tight_layout()
fig.savefig(analysis_dir / "test_horizon_errors.png", dpi=160)
plt.show()


## 3. 수요 규모별 7일 합계 오류

수요 규모 경계는 **Valid 기준선의 7일 예상 수요**로 정하고 Test에도 같은 값을 적용합니다. 실제 판매량은 구간 선택에 사용하지 않습니다. 그룹의 판매량 비중과 기준선 대비 추가 절대오차를 함께 봐서, 판매가 거의 없는 상품군의 큰 WAPE에 판단이 쏠리지 않게 합니다.


In [ ]:
valid_predictions = pd.read_csv(valid_paths["LightGBM"], parse_dates=["date", "target_date"])
valid_weekly = weekly_predictions(valid_predictions)
weekly = weekly_predictions(test_predictions)
cutoffs = valid_weekly["sales_same_weekday_4w_mean"].quantile([1 / 3, 2 / 3]).to_numpy()
valid_weekly = assign_demand_tier(valid_weekly, cutoffs)
weekly = assign_demand_tier(weekly, cutoffs)

valid_tier_scores = group_scores(valid_weekly, ["demand_tier"])
tier_scores = group_scores(weekly, ["demand_tier"])
weekly_family_scores = group_scores(weekly, ["family"])
weekly_store_scores = group_scores(weekly, ["store_nbr"])

for name, table in {
    "valid_7day_error_by_demand_tier": valid_tier_scores,
    "test_7day_error_by_demand_tier": tier_scores,
    "test_7day_error_by_family": weekly_family_scores,
    "test_7day_error_by_store": weekly_store_scores,
}.items():
    table.to_csv(analysis_dir / f"{name}.csv", index=False)

display(valid_tier_scores)
display(tier_scores)
display(weekly_family_scores.sort_values("excess_absolute_error", ascending=False).head(12))
display(weekly_store_scores.sort_values("excess_absolute_error", ascending=False).head(12))


## 4. 낮은 수요·취약 구간 진단

낮은 수요 구간에서 LightGBM이 기준선보다 얼마나 손해를 보는지 **추가 절대오차** 순서로 확인합니다. WAPE가 매우 큰 상품군이라도 실제 판매량 비중과 추가 절대오차가 작으면 전체 운영 영향은 제한적입니다. 이는 원인 탐색표이며, Test를 보고 제외 상품군을 확정하지 않습니다.

In [ ]:
low_weekly = weekly.loc[weekly["demand_tier"].eq("낮음")]
low_family = group_scores(low_weekly, ["family"])
low_store = group_scores(low_weekly, ["store_nbr"])

low_family.to_csv(analysis_dir / "test_low_demand_by_family.csv", index=False)
low_store.to_csv(analysis_dir / "test_low_demand_by_store.csv", index=False)
display(low_family.sort_values("excess_absolute_error", ascending=False).head(12))
display(low_store.sort_values("excess_absolute_error", ascending=False).head(12))

## 5. 기준선 대체 규칙 시험

Valid의 **낮은 수요 구간**에서 기준선이 더 정확했을 때만 해당 구간을 4주 기준선으로 대체합니다. 경계와 선택 여부를 Valid에서 정한 뒤 Test에 그대로 적용합니다. Test는 이미 여러 차례 확인했으므로 아래 Test 점수는 독립적인 성능 확증이 아니라 **탐색적 점검**입니다. 실제 적용 여부는 다른 과거 시점에서 다시 검증해야 합니다.

In [ ]:
valid_low = valid_tier_scores.loc[valid_tier_scores["demand_tier"].eq("낮음")].iloc[0]
use_low_fallback = bool(valid_low["excess_absolute_error"] > 0)

rule_rows = []
for phase, frame in [("Valid", valid_weekly), ("Test", weekly)]:
    fallback = frame["demand_tier"].eq("낮음") & use_low_fallback
    candidates = {
        "LightGBM": frame["prediction"],
        "4주 기준선": frame["sales_same_weekday_4w_mean"],
        "Valid 선택 규칙": frame["prediction"].where(
            ~fallback, frame["sales_same_weekday_4w_mean"]
        ),
    }
    for name, forecast in candidates.items():
        rule_rows.append({
            "phase": phase, "method": name, "rows": len(frame),
            "low_demand_fallback": use_low_fallback,
            **regression_metrics(frame["target_sales"], forecast),
        })

rule_scores = pd.DataFrame(rule_rows)
rule_scores.to_csv(analysis_dir / "weekly_fallback_exploration.csv", index=False)
display(rule_scores)

## 6. 다섯 모델의 같은 구간 비교

LightGBM·CatBoost·N-HiTS·TCN·TFT의 저장된 Valid/Test 예측에 **같은 4주 기준선과 Valid에서 정한 수요 경계**를 붙입니다. 7일 합계의 수요 규모·상품군·매장별 WAPE, Bias, 기준선 대비 추가 절대오차를 비교합니다. `gain_pp`가 음수이면 해당 모델이 기준선보다 나쁩니다. 이 셀은 모델을 다시 학습하지 않습니다.

In [ ]:
valid_model_weekly = model_weekly_predictions(valid_paths, valid_predictions, cutoffs)
test_model_weekly = model_weekly_predictions(test_paths, test_predictions, cutoffs)

all_tier_scores = model_group_scores(test_model_weekly, ["demand_tier"])
all_family_scores = model_group_scores(test_model_weekly, ["family"])
all_store_scores = model_group_scores(test_model_weekly, ["store_nbr"])

for name, table in {
    "all_models_test_7day_by_demand_tier": all_tier_scores,
    "all_models_test_7day_by_family": all_family_scores,
    "all_models_test_7day_by_store": all_store_scores,
}.items():
    table.to_csv(analysis_dir / f"{name}.csv", index=False)

display(all_tier_scores[[
    "model", "demand_tier", "rows", "actual_share_pct", "model_wape",
    "baseline_wape", "gain_pp", "model_bias",
]])
display(all_family_scores.sort_values("excess_absolute_error", ascending=False)
        .groupby("model", sort=False).head(5)[
            ["model", "family", "actual_sales", "model_wape", "baseline_wape",
             "excess_absolute_error"]
        ])
display(all_store_scores.sort_values("excess_absolute_error", ascending=False)
        .groupby("model", sort=False).head(5)[
            ["model", "store_nbr", "actual_sales", "model_wape", "baseline_wape",
             "excess_absolute_error"]
        ])

In [ ]:
heatmap = all_tier_scores.pivot(index="model", columns="demand_tier", values="gain_pp")
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(heatmap, annot=True, fmt=".1f", cmap="RdYlGn", center=0, ax=ax)
ax.set(xlabel="7일 예상 수요 규모", ylabel="", title="4주 기준선 대비 7일 WAPE 개선 (pp)")
fig.tight_layout()
fig.savefig(analysis_dir / "all_models_demand_tier_gain.png", dpi=160)
plt.show()

### 모델별 낮은 수요 대체 효과

각 모델에 대해 **그 모델의 Valid 낮은 수요**에서만 기준선이 더 정확한지 판단한 후, 선택한 규칙을 Test에 그대로 적용합니다. DL의 낮은 수요 오류를 고쳐도 전체 7일 WAPE가 기준선보다 높은지 확인하는 탐색적 비교입니다. Test는 이미 살펴본 구간이므로 여기서 선택한 모델이나 규칙의 독립적인 성능 증거로 사용하지 않습니다.

In [ ]:
all_rule_scores = model_fallback_scores(valid_model_weekly, test_model_weekly)
all_rule_scores.to_csv(analysis_dir / "all_models_weekly_fallback_exploration.csv", index=False)
display(all_rule_scores.loc[all_rule_scores["phase"].eq("Test"), [
    "model", "method", "low_demand_fallback", "rows", "rmsle", "mae",
    "rmse", "wape", "bias",
]].sort_values(["model", "method"]))

## 7. 주간 검토 후보

LightGBM의 7일 예상 수요와 4주 기준선의 차이로 검토 후보를 정렬합니다. 선택에 Test 정답을 쓰지 않으며, `target_sales`는 회고적 확인에만 표시합니다. 재고·공급 가능량이 없어 발주량을 제안하지 않습니다.


In [ ]:
weekly["forecast_change"] = weekly["prediction"] - weekly["sales_same_weekday_4w_mean"]
weekly["forecast_change_pct"] = 100 * weekly["forecast_change"] / weekly[
    "sales_same_weekday_4w_mean"
].replace(0, np.nan)

review_columns = [
    "date", "store_nbr", "family", "sales_same_weekday_4w_mean",
    "prediction", "forecast_change", "forecast_change_pct", "target_sales",
]
increasing_demand = weekly.nlargest(20, "forecast_change")[review_columns]
decreasing_demand = weekly.nsmallest(20, "forecast_change")[review_columns]

increasing_demand.to_csv(analysis_dir / "review_increasing_demand.csv", index=False)
decreasing_demand.to_csv(analysis_dir / "review_decreasing_demand.csv", index=False)
display(increasing_demand.head(10))
display(decreasing_demand.head(10))


### 한 기준일의 사례 비교

서로 겹치는 7일 예측을 여러 의사결정으로 세지 않도록 **Test의 마지막 기준일 한 번**만 사용합니다. 기준선 대비 변화가 큰 증가·감소 사례를 상품군별 한 건씩 골라 예측과 실제 7일 합계를 비교합니다. 사례 선택에는 실제 판매량을 쓰지 않습니다.

In [ ]:
one_cycle = weekly.loc[weekly["date"].eq(weekly["date"].max())].copy()
increasing_cases = one_cycle.sort_values("forecast_change", ascending=False).drop_duplicates("family").head(4)
decreasing_cases = one_cycle.sort_values("forecast_change").drop_duplicates("family").head(4)
cases = pd.concat([increasing_cases, decreasing_cases]).drop_duplicates("family")
cases["case"] = cases["store_nbr"].astype(str) + "호점 · " + cases["family"].astype(str)
cases[["date", "store_nbr", "family", "sales_same_weekday_4w_mean", "prediction",
       "target_sales", "forecast_change", "forecast_change_pct"]].to_csv(
    analysis_dir / "one_cycle_decision_cases.csv", index=False
)

case_plot = cases.melt(
    id_vars="case", value_vars=["sales_same_weekday_4w_mean", "prediction", "target_sales"],
    var_name="source", value_name="seven_day_sales",
)
case_plot["source"] = case_plot["source"].map({
    "sales_same_weekday_4w_mean": "4주 기준선", "prediction": "LightGBM", "target_sales": "실제",
})
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=case_plot, y="case", x="seven_day_sales", hue="source", ax=ax)
ax.set(xlabel="7일 합계 판매량", ylabel="", title=f"{one_cycle['date'].max():%Y-%m-%d} 기준일 검토 사례")
fig.tight_layout()
fig.savefig(analysis_dir / "one_cycle_decision_cases.png", dpi=160)
plt.show()
display(cases[["date", "store_nbr", "family", "forecast_change_pct", "target_sales"]])

## 8. 군집별 LightGBM 활용 범위

[02 수요 패턴 군집](02_demand_segments.ipynb)을 실행한 경우에만 표시됩니다. 군집은 Valid가 시작되기 전인 2017-07-18까지의 판매 이력으로 정의했습니다. 여기서는 저장된 같은 Test 행의 **7일 합계**를 붙여 군집별 LightGBM과 4주 기준선의 WAPE·Bias·판매량 비중을 비교합니다. 군집 번호는 품질 순위가 아닙니다.

In [ ]:
segment_path = REPO / "outputs" / "demand_segments" / "segment_features.parquet"
if segment_path.is_file():
    segment_labels = pd.read_parquet(
        segment_path, columns=["store_nbr", "family", "segment", "history_cutoff"],
    )
    assert pd.to_datetime(segment_labels["history_cutoff"]).eq("2017-07-18").all()
    assert not segment_labels.duplicated(["store_nbr", "family"]).any()
    segment_labels = segment_labels.drop(columns="history_cutoff")
    weekly_segments = weekly.merge(
        segment_labels, on=["store_nbr", "family"], how="left", validate="many_to_one",
    )
    weekly_segments["segment"] = weekly_segments["segment"].fillna("이력 부족")
    segment_scores = group_scores(weekly_segments, ["segment"])
    segment_scores.to_csv(analysis_dir / "test_7day_error_by_segment.csv", index=False)
    display(segment_scores[[
        "segment", "rows", "actual_share_pct", "lightgbm_wape", "baseline_wape",
        "gain_pp", "lightgbm_bias",
    ]].sort_values("segment"))

    chart = segment_scores.melt(
        id_vars="segment", value_vars=["lightgbm_wape", "baseline_wape"],
        var_name="method", value_name="wape",
    )
    chart["method"] = chart["method"].map({
        "lightgbm_wape": "LightGBM", "baseline_wape": "4주 기준선",
    })
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=chart, x="segment", y="wape", hue="method", ax=ax)
    ax.set(xlabel="수요 패턴 군집", ylabel="7일 WAPE (%)", title="군집별 기준선 대비 성능")
    fig.tight_layout()
    fig.savefig(analysis_dir / "test_wape_by_segment.png", dpi=160)
    plt.show()

## 해석 원칙과 다음 검증

- 전체 WAPE 개선과 **매장·상품군별 실패**를 함께 보고합니다. Bias가 양수이면 주간 총수요를 과대예측하는 방향입니다.
- 7일 합계 창이 서로 겹치므로 행 수만큼 독립적인 발주 사례를 검증한 것은 아닙니다. 마지막 기준일 시각화는 한 주기 사례입니다.
- 낮은 수요 대체 규칙은 Valid에서 정했지만 Test 결과를 이미 확인했습니다. [09 과거 시점 평가](09_lightgbm_historical_backtest.ipynb)에서 다른 날짜의 개선 반복 여부를 확인합니다.
- 다섯 모델의 수요 규모·상품군·매장별 결과는 같은 Test 행과 기준선으로 비교합니다. 모델별 입력 Feature가 달라 성능 차이를 구조만의 효과로 해석하지 않습니다.
- DL의 낮은 수요를 기준선으로 대체해도 높은 수요 구간이 기준선보다 나쁘면, 낮은 수요 오류만으로 전체 격차를 설명할 수 없습니다.
- 재고·공급·납기 자료가 없으므로 품절 감소, 비용 절감, 최적 발주량은 현재 결과로 검증할 수 없습니다.
- 군집별 Test 평가는 2017-07-18까지의 이력으로 만든 [02 수요 패턴 군집](02_demand_segments.ipynb)을 사용합니다. 군집은 학습 Feature가 아닙니다.
